# Fluoroscopy Simulation

Use generated CT scan volume as inpurt to the fluoroscopy simulation.

This script demonstrates the complete fluorosim workflow:
1. Load CT volume (DICOM, NIfTI, or cached μ-volume)
2. Initialize the GPU-accelerated simulator
3. Render a LAO/RAO C-arm sweep with accurate performance metrics

Features:
- GPU warmup for accurate FPS measurement (~150 FPS on RTX A6000)
- Automatic cache detection and fallback to synthetic volume
- Configurable geometry, realism, and output settings

Run with:
    cd fluoro-simulator
    python examples/fluorosim_demo.py

To clear cache and reload from source:
    rm -rf /tmp/fluoro_cache && python examples/fluorosim_demo.py

In [ ]:
from __future__ import annotations

import math

import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

from fluorosim import (
    CarmGeometry,
    FluoroSimulator,
    OutputSettings,
    Pose,
    PreprocessedVolume,
    RealismSettings,
    SimulatorConfig,
    VolumePreprocessor,
)

import stackview

# Configuration

Update these paths to your local data

In [ ]:
# Path to DICOM CT data (e.g., generated from NV-Generate-CTMR inference_tutorial script)
CT_BASENAME = "sample_20260403_141223_077521_image"
NIFTI_CT_FILE = NIFTI_CT_PATH = Path(CT_BASENAME + ".nii.gz")
NIFTI_CT_DIR = Path("~/workspace/NV-Generate-CTMR/output/").expanduser()
NIFTI_CT_PATH = NIFTI_CT_DIR / NIFTI_CT_FILE
print(f"Input CT file: {NIFTI_CT_PATH}")

_SCRIPT_DIR = Path("~/workspace/i4h-sensor-simulation/fluoro-simulator/examples").expanduser().resolve().parent.parent

# Output directory
OUTPUT_DIR = Path(os.environ.get("FLUOROSIM_OUTPUT_DIR", str(_SCRIPT_DIR / "output"))).expanduser()

# Cache directory for preprocessed volume
CACHE_DIR = Path(os.environ.get("FLUOROSIM_CACHE_DIR", str(OUTPUT_DIR / "cache" / CT_BASENAME))).expanduser()

# Create the cache directory if it doesn't exist
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Alternative: pre-generated mu_volume if it exists
CACHED_MU_VOLUME = CACHE_DIR / "mu_volume.npy"
print(f"Cache Directory: {CACHE_DIR}")

# Step 1

Load or preprocess the volume

In [ ]:
print("\n[Step 1] Loading CT Volume...")

# Check if we have a cached preprocessed volume
if (CACHE_DIR / "mu_volume.npy").exists():
    print(f"  Loading cached volume from: {CACHE_DIR}")
    volume = PreprocessedVolume.load(CACHE_DIR)

elif NIFTI_CT_PATH.exists():
    print(f"  Loading NIfTI from: {NIFTI_CT_PATH}")
    preprocessor = VolumePreprocessor.from_nifti(NIFTI_CT_PATH)
    print(f"  Shape: {preprocessor.shape}")
    print(f"  Spacing: {preprocessor.spacing_zyx_mm} mm")
    print(f"  HU range: {preprocessor.hu_range}")

    volume = preprocessor.preprocess(output_dir=CACHE_DIR)
else:
    print("\n  ERROR: CT data not found at:")
    print(f"    NIfTI: {NIFTI_CT_PATH}")
    print("  Please update the paths in this script.")
    print("\n  Alternatively, run with a synthetic volume:")
    print("    python -m ...run_imagecas --synthetic")

    # Create synthetic volume for demo
    print("\n  Creating synthetic sphere volume for demo...")
    import numpy as np
    shape = (128, 256, 256)
    z, y, x = np.ogrid[:shape[0], :shape[1], :shape[2]]
    center = np.array(shape) / 2
    dist = np.sqrt((z - center[0])**2 + (y - center[1])**2 + (x - center[2])**2)

    # Sphere with bone density in air
    hu_volume = np.where(dist < 50, 800.0, -900.0).astype(np.float32)

    preprocessor = VolumePreprocessor.from_numpy(
        hu_volume,
        spacing_zyx_mm=(1.0, 0.5, 0.5)
    )
    volume = preprocessor.preprocess(output_dir=CACHE_DIR)

print("\n  Preprocessed volume:")
print(f"    {volume}")

# Step 2

Create simulator with configuration

In [ ]:
print("\n[Step 2] Initializing Simulator...")

config = SimulatorConfig(
    geometry=CarmGeometry(
        detector_width_px=1024,
        detector_height_px=1024,
        pixel_spacing_mm=0.5,
        source_to_detector_mm=2220.0,
        source_to_isocenter_mm=2180.0,
    ),
    realism=RealismSettings(
        enabled=False,  # Disable noise for clean images
        # gaussian_sigma=0.015,
        # blur_sigma_px=0.3,
    ),
    output=OutputSettings(
        save_to_disk=True,
        output_dir=str(OUTPUT_DIR),
        format="png",
    ),
)

simulator = FluoroSimulator(volume, config)
print(f"  {simulator}")
print(f"  Backend: {type(simulator.renderer).__name__}")

# Step 3

Render a stack of chest x-ray images in a numpy array

In [ ]:
print("\n[Step 3] Rendering...")
stack = np.empty((0,1024,1024), dtype=np.float32)
for i in range(-36,36):
    frame = simulator.render_frame(rotation=(math.radians(-90), math.radians(180 + i*5), math.radians(0)), translation=(0, 0, 0))
    stack = np.append(stack, frame.image[np.newaxis, :, :], axis=0)

# Step 4

Visualize the stack using the stackview widget

In [ ]:
stackview.slice(stack, continuous_update=True)